# Step 2 — Naive Bayes Sentiment Model

Project 2 (HPDP, SECP3133) — Real-Time Sentiment Analysis on Grab Google Play Reviews

**Owner:** Syahmi (NLP & Model Engineer) · **Model category:** Machine Learning (§6.3)

Baseline model: **TF-IDF features + Multinomial Naive Bayes**.

Why this is the baseline:
- Fastest of the three models (trains in seconds on CPU)
- Simple, interpretable, well-understood probabilistic classifier
- Gives a floor that LSTM and DistilBERT must beat to justify their complexity

Handles the **severe class imbalance** (Neutral is ~5%) via class-balanced sample weights.
Headline metric is **macro-F1**, not accuracy.

## 0. Setup + load splits

Reads `train` and `test` from Step 1. Auto-detects `.csv` or `.xlsx`, and works on Colab or locally.

In [ ]:
import os, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score)

LABELS = ['Negative', 'Neutral', 'Positive']  # index = sentiment_label 0/1/2

def find_file(stem):
    """Locate train/val/test, csv or xlsx, in common Colab + local locations.
    Includes the Colab root (/content) where uploaded files land by default."""
    roots = ['.', 'data', '/content', '/content/data', '../data']
    for r in roots:
        for ext in ('csv', 'xlsx'):
            p = f'{r}/{stem}.{ext}'
            if os.path.exists(p):
                return p
    raise FileNotFoundError(
        f'{stem}.(csv|xlsx) not found. On Colab, upload {stem}.csv to the file '
        'panel (it lands in /content/), then re-run this cell.')

def load(stem):
    p = find_file(stem)
    df = pd.read_excel(p) if p.endswith('.xlsx') else pd.read_csv(p)
    df['final_clean_text'] = df['final_clean_text'].fillna('')
    print(f'  {stem}: {len(df):,} rows  <- {p}')
    return df

MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

print('Loading splits...')
train = load('train')
test  = load('test')

## 1. TF-IDF vectorization

Converts cleaned text into numeric feature vectors. Fit the vectorizer on **train only**, then
transform test with the same vocabulary (never fit on test — that leaks information).

- `max_features=10000` — cap vocabulary to the 10k most informative terms
- `ngram_range=(1,2)` — unigrams + bigrams, so phrases like "not good" are captured
- `min_df=2` — ignore terms appearing in only one review (noise)

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)

X_train = vectorizer.fit_transform(train['final_clean_text'])
X_test  = vectorizer.transform(test['final_clean_text'])
y_train = train['sentiment_label'].astype(int)
y_test  = test['sentiment_label'].astype(int)

print(f'X_train: {X_train.shape}  (reviews x features)')
print(f'X_test:  {X_test.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')

## 2. Train with class-balanced weights

`MultinomialNB` has no `class_weight` parameter, so we pass **sample weights** computed with
`'balanced'` — this up-weights the rare Neutral class so the model doesn't ignore it.

In [ ]:
sample_weights = compute_sample_weight('balanced', y_train)

model = MultinomialNB()
t0 = time.time()
model.fit(X_train, y_train, sample_weight=sample_weights)
train_time = time.time() - t0

print(f'Trained in {train_time:.2f} s')

## 3. Evaluate on the test set

Required metrics (§6.3): accuracy, precision, recall, F1, confusion matrix.
Watch the **Neutral** row — that's where imbalance hurts most.

In [ ]:
t0 = time.time()
y_pred = model.predict(X_test)
infer_time = time.time() - t0
infer_ms = infer_time / len(y_test) * 1000

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f'Accuracy : {acc:.4f}')
print(f'Macro-F1 : {macro_f1:.4f}')
print(f'Inference: {infer_ms:.4f} ms/sample\n')
print(classification_report(y_test, y_pred, target_names=LABELS, digits=4))

## 4. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Naive Bayes — Confusion Matrix\nAccuracy={acc:.3f}  Macro-F1={macro_f1:.3f}')
plt.tight_layout()
plt.savefig('nb_confusion_matrix.png', dpi=120)
plt.show()

## 5. Inspect most informative words per class

Sanity check: do the top words make intuitive sense? Good material for the report.

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
for idx, label in enumerate(LABELS):
    top = np.argsort(model.feature_log_prob_[idx])[-15:][::-1]
    print(f'{label:9s}: ' + ', '.join(feature_names[top]))

## 6. Save model + vectorizer + metrics

Both files are required at inference time — the vectorizer must match the model's vocabulary.
The metrics JSON is consumed by Step 5 (model comparison).

In [ ]:
import json

joblib.dump(model, f'{MODELS_DIR}/naive_bayes.joblib')
joblib.dump(vectorizer, f'{MODELS_DIR}/tfidf_vectorizer.joblib')

report = classification_report(y_test, y_pred, target_names=LABELS,
                               output_dict=True)
metrics = {
    'model': 'Naive Bayes',
    'accuracy': acc,
    'macro_f1': macro_f1,
    'macro_precision': report['macro avg']['precision'],
    'macro_recall': report['macro avg']['recall'],
    'f1_negative': report['Negative']['f1-score'],
    'f1_neutral': report['Neutral']['f1-score'],
    'f1_positive': report['Positive']['f1-score'],
    'train_time_s': train_time,
    'inference_ms_per_sample': infer_ms,
}
with open(f'{MODELS_DIR}/metrics_naive_bayes.json', 'w') as f:
    json.dump(metrics, f, indent=2)

size_mb = (os.path.getsize(f'{MODELS_DIR}/naive_bayes.joblib') +
           os.path.getsize(f'{MODELS_DIR}/tfidf_vectorizer.joblib')) / 1e6
print(f'Saved naive_bayes.joblib + tfidf_vectorizer.joblib ({size_mb:.1f} MB total)')
print(f'Saved metrics_naive_bayes.json')
print('\n[OK] Step 2 complete.')

## 7. (Colab) Download outputs to your computer

Run this to download the trained model, vectorizer, metrics, and confusion matrix
straight to your browser's Downloads folder. Move them into your local
`models/` and `figures/` folders afterwards.

In [ ]:
from google.colab import files

for f in ['naive_bayes.joblib', 'tfidf_vectorizer.joblib', 'metrics_naive_bayes.json']:
    files.download(f'{MODELS_DIR}/{f}')
files.download('nb_confusion_matrix.png')